# 01 — Dataset Acquisition, Comparison & Audit (NSL-KDD vs. KDD Cup 99)

This notebook performs a programmatic data quality audit of the **NSL-KDD** benchmark (`KDDTrain+.TXT` and `KDDTest+.TXT`) and compares it against its predecessor **KDD Cup 99**.

**Audit Checklist:**
1. Candidate dataset comparison (NSL-KDD vs. KDD Cup 99)
2. File shapes, column verification (41 features + `attack_type` + `difficulty`), and data types
3. Missing values and duplicate checks (within-file and train/test overlap)
4. Class balance (`normal` vs. `attack`) and attack-type distribution
5. Unseen attack types and unseen categorical values (`service`, `protocol_type`, `flag`) in `KDDTest+`
6. Constant columns, privacy, and benchmark bias assessment

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Allow importing reusable project modules from src/
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    CATEGORICAL_COLUMNS,
    DIFFICULTY_COLUMN,
    FEATURE_COLUMNS,
    NUMERIC_COLUMNS,
    RAW_COLUMNS,
    TEST_FILE,
    TRAIN_FILE,
)
from src.load_data import load_train_test_splits

print(f"Project directory: {PROJECT_ROOT.name}")
print(f"Verified feature count: {len(FEATURE_COLUMNS)} ({len(CATEGORICAL_COLUMNS)} categorical, {len(NUMERIC_COLUMNS)} numeric)")
print(f"Expected raw column count: {len(RAW_COLUMNS)}")

Project directory: network-intrusion-detection-prototype
Verified feature count: 41 (3 categorical, 38 numeric)
Expected raw column count: 43


## 1. Candidate Dataset Comparison: NSL-KDD vs. KDD Cup 99

Before committing to NSL-KDD, we compare it with its parent dataset **KDD Cup 99** (full statistics from UNB / Tavallaee et al., 2009, plus an empirical duplicate check on KDD Cup 99's 10% subset when cached locally).

In [2]:
comparison_df = pd.DataFrame([
    {
        "Criterion": "Training Records",
        "NSL-KDD (Selected)": "125,973 (KDDTrain+)",
        "KDD Cup 99 (Rejected)": "4,898,431 (1,074,992 distinct)",
    },
    {
        "Criterion": "Test Records",
        "NSL-KDD (Selected)": "22,544 (KDDTest+)",
        "KDD Cup 99 (Rejected)": "311,027 (77,289 distinct)",
    },
    {
        "Criterion": "Duplicate Rate",
        "NSL-KDD (Selected)": "0 exact full-row duplicates within train/test",
        "KDD Cup 99 (Rejected)": "~78% full train / ~75% full test (70.53% verified on 10% subset via sklearn)",
    },
    {
        "Criterion": "Compute Feasibility",
        "NSL-KDD (Selected)": "Runs full splits locally without downsampling",
        "KDD Cup 99 (Rejected)": "Requires heavy sampling to train iteratively",
    },
    {
        "Criterion": "Licence & Citation",
        "NSL-KDD (Selected)": "Free to redistribute/mirror; cite Tavallaee et al. (2009)",
        "KDD Cup 99 (Rejected)": "Public research benchmark (UCI / DARPA 1998)",
    },
    {
        "Criterion": "Selection Decision",
        "NSL-KDD (Selected)": "SELECTED",
        "KDD Cup 99 (Rejected)": "REJECTED (severe redundancy and scale)",
    },
])
print(comparison_df.to_string(index=False))

          Criterion                                        NSL-KDD (Selected)                                                        KDD Cup 99 (Rejected)
   Training Records                                       125,973 (KDDTrain+)                                               4,898,431 (1,074,992 distinct)
       Test Records                                         22,544 (KDDTest+)                                                    311,027 (77,289 distinct)
     Duplicate Rate             0 exact full-row duplicates within train/test ~78% full train / ~75% full test (70.53% verified on 10% subset via sklearn)
Compute Feasibility             Runs full splits locally without downsampling                                 Requires heavy sampling to train iteratively
 Licence & Citation Free to redistribute/mirror; cite Tavallaee et al. (2009)                                 Public research benchmark (UCI / DARPA 1998)
 Selection Decision                                                  S

## 2. Load NSL-KDD Splits & Verify Schema / Dtypes

We load both headerless files using `src.load_data.load_train_test_splits()`, which verifies file existence, non-emptiness, exact 43-column width, numeric type integrity, and attaches binary `label` (`0 = normal`, `1 = attack`) and `attack_family`.

In [3]:
train_df, test_df = load_train_test_splits(TRAIN_FILE, TEST_FILE)

print(f"KDDTrain+ shape (with label & attack_family): {train_df.shape}")
print(f"KDDTest+  shape (with label & attack_family): {test_df.shape}")
print(f"Raw feature + metadata columns: {len(RAW_COLUMNS)}")
print("\nDtype counts (41 predictive features):")
print(train_df[FEATURE_COLUMNS].dtypes.value_counts())
print("\nFirst 3 rows (subset of columns):")
print(train_df[["duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "attack_type", "difficulty", "label"]].head(3).to_string(index=False))

KDDTrain+ shape (with label & attack_family): (125973, 45)
KDDTest+  shape (with label & attack_family): (22544, 45)
Raw feature + metadata columns: 43

Dtype counts (41 predictive features):
int64      23
float64    15
str         3
Name: count, dtype: int64

First 3 rows (subset of columns):
 duration protocol_type  service flag  src_bytes  dst_bytes attack_type  difficulty  label
        0           tcp ftp_data   SF        491          0      normal          20      0
        0           udp    other   SF        146          0      normal          15      0
        0           tcp  private   S0          0          0     neptune          19      1


## 3. Missing Values & Duplicate Audit

We audit missing values per column and check duplicates at three levels: exact full rows, `(41 features + attack_type)`, and `41 features` alone, both within each file and across the train/test boundary.

In [4]:
train_missing = int(train_df[RAW_COLUMNS].isna().sum().sum())
test_missing = int(test_df[RAW_COLUMNS].isna().sum().sum())
print(f"Total missing values — Train: {train_missing}, Test: {test_missing}")

train_feat_hash = pd.util.hash_pandas_object(train_df[FEATURE_COLUMNS], index=False)
test_feat_hash = pd.util.hash_pandas_object(test_df[FEATURE_COLUMNS], index=False)
train_fa_hash = pd.util.hash_pandas_object(train_df[FEATURE_COLUMNS + ["attack_type"]], index=False)
test_fa_hash = pd.util.hash_pandas_object(test_df[FEATURE_COLUMNS + ["attack_type"]], index=False)

dup_report = pd.DataFrame([
    {"Check": "Exact full-row duplicates (43 raw cols)", "Train": int(train_df[RAW_COLUMNS].duplicated().sum()), "Test": int(test_df[RAW_COLUMNS].duplicated().sum())},
    {"Check": "Feature + attack_type duplicates (42 cols)", "Train": int(train_df.duplicated(subset=FEATURE_COLUMNS + ["attack_type"]).sum()), "Test": int(test_df.duplicated(subset=FEATURE_COLUMNS + ["attack_type"]).sum())},
    {"Check": "Feature-only duplicates (41 cols)", "Train": int(train_df.duplicated(subset=FEATURE_COLUMNS).sum()), "Test": int(test_df.duplicated(subset=FEATURE_COLUMNS).sum())},
])
print("\nWithin-file duplicate summary:")
print(dup_report.to_string(index=False))

print(f"\nCross-file overlap (unique 41-feature vectors in both train & test): {len(set(train_feat_hash) & set(test_feat_hash))}")
print(f"Cross-file overlap (test rows whose 41 features appear in train):  {int(test_feat_hash.isin(set(train_feat_hash)).sum())}")
print(f"Cross-file overlap (unique feature+attack rows in both files):     {len(set(train_fa_hash) & set(test_fa_hash))}")

Total missing values — Train: 0, Test: 0

Within-file duplicate summary:
                                     Check  Train  Test
   Exact full-row duplicates (43 raw cols)      0     0
Feature + attack_type duplicates (42 cols)      0     0
         Feature-only duplicates (41 cols)     16    57

Cross-file overlap (unique 41-feature vectors in both train & test): 654
Cross-file overlap (test rows whose 41 features appear in train):  664
Cross-file overlap (unique feature+attack rows in both files):     610


## 4. Class Balance, Attack Families & Unseen Test Attacks

A defining property of NSL-KDD is that `KDDTest+` intentionally includes novel attack types never observed in `KDDTrain+`. Quantifying them here explains why test recall is expected to drop below validation recall.

In [5]:
for name, df in [("KDDTrain+", train_df), ("KDDTest+", test_df)]:
    vc = df["label"].value_counts().sort_index()
    pct = (vc / len(df) * 100).round(2)
    print(f"{name}: Normal (0) = {vc[0]:,} ({pct[0]}%) | Attack (1) = {vc[1]:,} ({pct[1]}%)")

print("\nAttack Family Counts:")
fam_df = pd.DataFrame({
    "Train Count": train_df["attack_family"].value_counts(),
    "Test Count": test_df["attack_family"].value_counts(),
}).fillna(0).astype(int)
print(fam_df)

train_attacks = set(train_df["attack_type"].unique()) - {"normal"}
test_attacks = set(test_df["attack_type"].unique()) - {"normal"}
unseen_attacks = sorted(test_attacks - train_attacks)
train_only_attacks = sorted(train_attacks - test_attacks)

unseen_mask = test_df["attack_type"].isin(unseen_attacks)
unseen_count = int(unseen_mask.sum())
test_attack_total = int((test_df["label"] == 1).sum())

print(f"\nUnique attack types — Train: {len(train_attacks)}, Test: {len(test_attacks)}")
print(f"Unseen attack types present ONLY in KDDTest+ ({len(unseen_attacks)}): {unseen_attacks}")
print(f"Test rows belonging to unseen attacks: {unseen_count:,} ({unseen_count/len(test_df)*100:.2f}% of all test rows; {unseen_count/test_attack_total*100:.2f}% of test attacks)")
print(f"Attack types present ONLY in KDDTrain+ ({len(train_only_attacks)}): {train_only_attacks}")
print("\nUnseen attack row counts in KDDTest+:")
print(test_df.loc[unseen_mask, "attack_type"].value_counts().to_string())

KDDTrain+: Normal (0) = 67,343 (53.46%) | Attack (1) = 58,630 (46.54%)
KDDTest+: Normal (0) = 9,711 (43.08%) | Attack (1) = 12,833 (56.92%)

Attack Family Counts:
               Train Count  Test Count
attack_family                         
DoS                  45927        7460
Normal               67343        9711
Probe                11656        2421
R2L                    995        2885
U2R                     52          67

Unique attack types — Train: 22, Test: 37
Unseen attack types present ONLY in KDDTest+ (17): ['apache2', 'httptunnel', 'mailbomb', 'mscan', 'named', 'processtable', 'ps', 'saint', 'sendmail', 'snmpgetattack', 'snmpguess', 'sqlattack', 'udpstorm', 'worm', 'xlock', 'xsnoop', 'xterm']
Test rows belonging to unseen attacks: 3,750 (16.63% of all test rows; 29.22% of test attacks)
Attack types present ONLY in KDDTrain+ (2): ['spy', 'warezclient']

Unseen attack row counts in KDDTest+:
attack_type
mscan            996
apache2          737
processtable     685
snmp

## 5. Categorical Cardinality, Unseen Categories & Constant Features

Checking category mismatches between train and test confirms why `OneHotEncoder(handle_unknown="ignore")` is mandatory for `service`, `protocol_type`, and `flag`, and verifies which constant columns carry zero variance.

In [6]:
for col in CATEGORICAL_COLUMNS:
    tr_cats = set(train_df[col].unique())
    te_cats = set(test_df[col].unique())
    print(f"{col}: train={len(tr_cats)}, test={len(te_cats)}")
    print(f"  In train only ({len(tr_cats - te_cats)}): {sorted(tr_cats - te_cats)}")
    print(f"  In test only  ({len(te_cats - tr_cats)}): {sorted(te_cats - tr_cats)}")

const_train = [c for c in FEATURE_COLUMNS if train_df[c].nunique() <= 1]
const_test = [c for c in FEATURE_COLUMNS if test_df[c].nunique() <= 1]
print(f"\nConstant columns in KDDTrain+: {const_train}")
print(f"Constant columns in KDDTest+:  {const_test}")
print("\nDifficulty column range:")
print(f"  Train difficulty: min={train_df[DIFFICULTY_COLUMN].min()}, max={train_df[DIFFICULTY_COLUMN].max()}, median={train_df[DIFFICULTY_COLUMN].median()}")
print(f"  Test  difficulty: min={test_df[DIFFICULTY_COLUMN].min()}, max={test_df[DIFFICULTY_COLUMN].max()}, median={test_df[DIFFICULTY_COLUMN].median()}")

protocol_type: train=3, test=3
  In train only (0): []
  In test only  (0): []
service: train=70, test=64
  In train only (6): ['aol', 'harvest', 'http_2784', 'http_8001', 'red_i', 'urh_i']
  In test only  (0): []
flag: train=11, test=11
  In train only (0): []
  In test only  (0): []

Constant columns in KDDTrain+: ['num_outbound_cmds']
Constant columns in KDDTest+:  ['num_outbound_cmds']

Difficulty column range:
  Train difficulty: min=0, max=21, median=20.0
  Test  difficulty: min=0, max=21, median=20.0


## 6. Key Audit Conclusions (Privacy, Bias & Modeling Implications)

- **Clean schema, zero missing values:** Both `KDDTrain+` (`125,973 x 43`) and `KDDTest+` (`22,544 x 43`) have **0 missing values** and **0 exact full-row duplicates** within each file.
- **Distribution shift & unseen attacks:** Attacks comprise **46.54%** of `KDDTrain+` versus **56.92%** of `KDDTest+`. Moreover, **17 attack types** (3,750 rows, or **29.22%** of all test attacks) never appear in `KDDTrain+`.
- **Category coverage:** `service` has **70** categories in train vs. **64** in test (with `6` train-only services: `['aol', 'harvest', 'http_2784', 'http_8001', 'red_i', 'urh_i']`). Even though `KDDTest+` has no unseen services here, real-world/uploaded CSVs can contain new services, so `OneHotEncoder(handle_unknown="ignore")` remains mandatory.
- **Constant feature:** `['num_outbound_cmds']` is constant (`0` everywhere) across both train and test and will be removed in feature selection (Experiment E2).
- **Drop `difficulty`:** `difficulty` encodes how many of 21 benchmark classifiers got the record right—not a live network attribute—so it is strictly excluded from model features.
- **Privacy & Bias:** NSL-KDD contains no IP addresses, usernames, or PII, but because it originates from simulated 1998-era traffic, benchmark metrics cannot be extrapolated to modern enterprise networks without fresh local validation and privacy review.